In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

In [7]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [8]:
df_train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [9]:
print(df_train.shape)
print(df_test.shape)

(691369, 14)
(296302, 13)


In [10]:
print(df_train.info())
print(df_test.info())

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  str    
 11  stress_level             636221 non-null  str    
 12  academic_work_impact     647145 non-null  str    
 13  addicted_label           691369 non-null  int64  
dtypes: float64(9), 

In [11]:
print(df_train.isnull().sum())
print("________________________________________")
print(df_test.isnull().sum())

id                              0
age                         28929
daily_screen_time_hours     95854
social_media_hours         133995
gaming_hours               126821
work_study_hours            51518
sleep_hours                 44480
notifications_per_day       67584
app_opens_per_day           80710
weekend_screen_time        112063
gender                      29034
stress_level                55148
academic_work_impact        44224
addicted_label                  0
dtype: int64
________________________________________
id                             0
age                        17138
daily_screen_time_hours    32788
social_media_hours         47397
gaming_hours               59420
work_study_hours           27777
sleep_hours                22455
notifications_per_day      34221
app_opens_per_day          25705
weekend_screen_time        50697
gender                     14212
stress_level               19626
academic_work_impact       25721
dtype: int64


In [12]:
print("Duplicate rows in train part:", df_train.duplicated().sum())
print("Duplicate rows in test part:", df_test.duplicated().sum())

Duplicate rows in train part: 0
Duplicate rows in test part: 0


In [13]:

X = df_train.drop(columns=['id' , 'addicted_label'])

y = df_train['addicted_label']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [15]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

Numeric columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']
Categorical columns: ['gender', 'stress_level', 'academic_work_impact']


C:\Users\Mithun\AppData\Local\Temp\ipykernel_23392\746075340.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [16]:
num_cols = ['age','daily_screen_time_hours','social_media_hours','gaming_hours',
            'work_study_hours','sleep_hours','notifications_per_day',
            'app_opens_per_day','weekend_screen_time']
cat_cols = ['gender','stress_level','academic_work_impact']

In [17]:
for col in num_cols + cat_cols:
    X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
    X_test[f'{col}_missing']  = X_test[col].isna().astype(int)
    df_test[f'{col}_missing'] = df_test[col].isna().astype(int)

In [18]:
#Impute numeric columns (fit on X_train only)
num_imputer = SimpleImputer(strategy='median')

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols]  = num_imputer.transform(X_test[num_cols])
df_test[num_cols] = num_imputer.transform(df_test[num_cols])

#Impute categorical columns (fit on X_train only)
cat_imputer = SimpleImputer(strategy='most_frequent')

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols]  = cat_imputer.transform(X_test[cat_cols])
df_test[cat_cols] = cat_imputer.transform(df_test[cat_cols])

# Sanity check — should all be 0 now
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum(), df_test.isnull().sum().sum())

0 0 0


In [19]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 553095 entries, 563697 to 685287
Data columns (total 24 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   age                              553095 non-null  float64
 1   daily_screen_time_hours          553095 non-null  float64
 2   social_media_hours               553095 non-null  float64
 3   gaming_hours                     553095 non-null  float64
 4   work_study_hours                 553095 non-null  float64
 5   sleep_hours                      553095 non-null  float64
 6   notifications_per_day            553095 non-null  float64
 7   app_opens_per_day                553095 non-null  float64
 8   weekend_screen_time              553095 non-null  float64
 9   gender                           553095 non-null  str    
 10  stress_level                     553095 non-null  str    
 11  academic_work_impact             553095 non-null  str    
 12  age_missing  

In [20]:
for col in cat_cols:
    print(col, X_train[col].unique())

gender <StringArray>
['Female', 'Other', 'Male']
Length: 3, dtype: str
stress_level <StringArray>
['Medium', 'High', 'Low']
Length: 3, dtype: str
academic_work_impact <StringArray>
['Yes', 'No']
Length: 2, dtype: str


In [21]:
# 1. stress_level — ordinal, explicit order Low < Medium < High
stress_order = ['Low', 'Medium', 'High']
ordinal_enc = OrdinalEncoder(categories=[stress_order])

X_train['stress_level'] = ordinal_enc.fit_transform(X_train[['stress_level']])
X_test['stress_level']  = ordinal_enc.transform(X_test[['stress_level']])
df_test['stress_level'] = ordinal_enc.transform(df_test[['stress_level']])

# 2. academic_work_impact — binary Yes/No -> 1/0
mapping = {'Yes': 1, 'No': 0}
X_train['academic_work_impact'] = X_train['academic_work_impact'].map(mapping)
X_test['academic_work_impact']  = X_test['academic_work_impact'].map(mapping)
df_test['academic_work_impact'] = df_test['academic_work_impact'].map(mapping)

# 3. gender — nominal, one-hot encode
X_train = pd.get_dummies(X_train, columns=['gender'], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=['gender'], drop_first=True)
df_test = pd.get_dummies(df_test, columns=['gender'], drop_first=True)

# Align columns in case one-hot produced mismatched columns between sets
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
df_test = df_test.reindex(columns=X_train.columns, fill_value=0)

In [22]:
print(X_train.dtypes)
print(X_train.isnull().sum().sum())  # should be 0
X_train.head()

age                                float64
daily_screen_time_hours            float64
social_media_hours                 float64
gaming_hours                       float64
work_study_hours                   float64
sleep_hours                        float64
notifications_per_day              float64
app_opens_per_day                  float64
weekend_screen_time                float64
stress_level                       float64
academic_work_impact                 int64
age_missing                          int64
daily_screen_time_hours_missing      int64
social_media_hours_missing           int64
gaming_hours_missing                 int64
work_study_hours_missing             int64
sleep_hours_missing                  int64
notifications_per_day_missing        int64
app_opens_per_day_missing            int64
weekend_screen_time_missing          int64
gender_missing                       int64
stress_level_missing                 int64
academic_work_impact_missing         int64
gender_Male

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,...,work_study_hours_missing,sleep_hours_missing,notifications_per_day_missing,app_opens_per_day_missing,weekend_screen_time_missing,gender_missing,stress_level_missing,academic_work_impact_missing,gender_Male,gender_Other
563697,25.0,6.74,2.31,1.33,2.20,6.23,223.0,169.0,10.14,1.0,...,1,0,0,0,0,0,0,1,False,False
673485,35.0,7.77,1.37,1.00,1.35,6.30,150.0,104.0,5.67,1.0,...,0,0,1,1,0,0,0,0,False,True
440270,20.0,7.79,2.42,0.22,4.66,6.90,186.0,104.0,7.96,2.0,...,0,0,0,1,0,0,0,0,True,False
378032,18.0,3.46,1.17,0.69,2.20,5.58,142.0,140.0,5.46,1.0,...,1,0,0,0,0,0,0,0,True,False
446226,29.0,7.77,2.19,1.35,3.73,8.78,197.0,121.0,10.58,2.0,...,0,0,0,0,0,0,0,0,True,False


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score


In [24]:
# Fix bool -> int first (if not done already)
bool_cols = X_train.select_dtypes(include='bool').columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols]  = X_test[bool_cols].astype(int)
df_test[bool_cols] = df_test[bool_cols].astype(int)

# Define parameter distribution to sample from
param_dist = {
    'n_estimators': randint(200, 600),
    'max_depth': [10, 20, 30, 40, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', 0.5],
    'class_weight': ['balanced', None]
}

# n_jobs=1 here — don't parallelize at the estimator level too,
# or it'll oversubscribe your CPU threads against the search's own n_jobs=-1
rf = RandomForestClassifier(random_state=42, n_jobs=1)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,              # number of random combinations to try — raise for more thorough search
    scoring='roc_auc',      # optimize for ROC-AUC; change to 'accuracy' or 'f1' if preferred
    cv=3,                   # 3-fold CV — keep low given dataset size (~691k rows), or it'll be slow
    verbose=2,
    random_state=42,
    n_jobs=-1               # parallelize across the n_iter x cv fits instead
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_)

best_rf = random_search.best_estimator_

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'class_weight': None, 'max_depth': 40, 'max_features': 0.5, 'min_samples_leaf': 7, 'min_samples_split': 13, 'n_estimators': 463}
Best CV ROC-AUC: 0.9422493525261051


In [26]:
best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.8694331544614317
Test ROC-AUC: 0.9422155613809657

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.74      0.77     40179
           1       0.90      0.92      0.91     98095

    accuracy                           0.87    138274
   macro avg       0.85      0.83      0.84    138274
weighted avg       0.87      0.87      0.87    138274


Confusion Matrix:
 [[29737 10442]
 [ 7612 90483]]
